# Conversation Memory and the Multi-Turn Session Loop

**Tutorial reference:** Part 6 — Layer 5 of the SmartIntake Learning Tutorial

This notebook covers the final layer: maintaining conversation state across turns and routing each session through the correct path (happy, partial, or fallback).

The key rule: **once a field is provided by the user, it must never be asked for again.**

---

## 1. Setup

In [ ]:
from dotenv import load_dotenv
import os
import json
from datetime import datetime
from pydantic import BaseModel, field_validator, ValidationError
from typing import Literal, Optional
from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory

load_dotenv()

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
    max_tokens=512
)

os.makedirs("output", exist_ok=True)
print("Setup complete.")


## 2. The IntakeRecord Model

Copied from NB-04 — the canonical Pydantic model for the project.

In [ ]:
class IntakeRecord(BaseModel):
    query_type: Literal[
        "complaint", "submission", "variation", "safety_signal",
        "label_update", "inspection", "general_enquiry"
    ]
    regulation_ref: Literal[
        "FDA_21CFR", "EMA_CTR", "ICH_E2A", "ICH_Q10",
        "CDSCO_NDC", "GxP_GMP", "GxP_GCP", "other"
    ]
    product_area: Literal[
        "oncology", "cardiovascular", "infectious_disease",
        "cmc", "clinical", "labelling", "general"
    ]
    urgency: Literal["routine", "standard", "urgent", "critical"]
    submitting_team: str

    @field_validator("submitting_team")
    @classmethod
    def team_must_not_be_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("submitting_team cannot be empty")
        return v.strip().title()

    @field_validator("submitting_team")
    @classmethod
    def team_not_a_person(cls, v: str) -> str:
        tokens = v.strip().split()
        if len(tokens) == 1 and tokens[0][0].isupper() and tokens[0].isalpha():
            raise ValueError(
                "submitting_team appears to be an individual name. "
                "Please provide a team or function name."
            )
        return v


print("IntakeRecord model ready.")

## 3. ConversationBufferMemory

`ConversationBufferMemory` stores every message verbatim. It is the right choice for short sessions (up to 10 turns) where full fidelity matters more than token efficiency.

In [ ]:
# Demonstrate how ConversationBufferMemory accumulates messages.
# In the real session loop, we inject the memory into each API call as context.

buffer_memory = ConversationBufferMemory(return_messages=True)

# Simulate adding turns to the memory
buffer_memory.chat_memory.add_user_message(
    "Hi, we need to file a Type II variation for our cardiovascular product with the EMA."
)
buffer_memory.chat_memory.add_ai_message(
    "I have noted the variation type and regulation. Could you tell me the urgency level "
    "— is there a specific regulatory deadline for this filing?"
)
buffer_memory.chat_memory.add_user_message("The deadline is in about 3 months — standard priority.")

# Load the messages — this is what we inject into the next API call
history = buffer_memory.load_memory_variables({})
print("Buffer memory contents after 3 turns:")
for msg in history["history"]:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"  [{role}] {msg.content[:80]}")

## 4. ConversationSummaryMemory

`ConversationSummaryMemory` compresses the conversation into a running summary after each turn. Use this when sessions exceed 10 turns to keep token usage manageable.

In [ ]:
# ConversationSummaryMemory uses the LLM itself to generate the summary.
# It requires the llm parameter at construction time.

summary_memory = ConversationSummaryMemory(llm=llm, return_messages=False)

# Add the same three turns as above
summary_memory.save_context(
    {"input": "Hi, we need to file a Type II variation for our cardiovascular product with the EMA."},
    {"output": "I have noted the variation type. Could you confirm the urgency and submitting team?"}
)
summary_memory.save_context(
    {"input": "The deadline is in about 3 months — standard priority."},
    {"output": "Thank you. Could you confirm which team is submitting this query?"}
)

summary = summary_memory.load_memory_variables({})
print("Summary memory after 2 exchanges:")
print(summary["history"])

## 5. The Memory Switching Logic

Use buffer memory for sessions up to 10 turns; switch to summary memory beyond that.

In [ ]:
# Memory manager that switches between buffer and summary based on turn count.
# In a real implementation this would transfer the existing history on switch.

BUFFER_LIMIT = 10

class MemoryManager:
    """
    Manages conversation memory with automatic switching at BUFFER_LIMIT turns.
    
    Uses ConversationBufferMemory for short sessions (full fidelity).
    Switches to ConversationSummaryMemory for long sessions (token efficiency).
    """

    def __init__(self, llm):
        self.llm = llm
        self.turn_count = 0
        self._memory = ConversationBufferMemory(return_messages=True)
        self._using_summary = False

    def add_turn(self, user_message: str, assistant_response: str):
        """Record a completed turn and switch memory type if the limit is reached."""
        self.turn_count += 1

        # Switch to summary memory after BUFFER_LIMIT turns
        if self.turn_count == BUFFER_LIMIT and not self._using_summary:
            print(f"[Memory] Switching to ConversationSummaryMemory at turn {self.turn_count}.")
            self._memory = ConversationSummaryMemory(llm=self.llm, return_messages=False)
            self._using_summary = True

        if self._using_summary:
            self._memory.save_context(
                {"input": user_message},
                {"output": assistant_response}
            )
        else:
            self._memory.chat_memory.add_user_message(user_message)
            self._memory.chat_memory.add_ai_message(assistant_response)

    def get_history_text(self) -> str:
        """Return the conversation history as a plain text string for prompt injection."""
        history = self._memory.load_memory_variables({})
        if self._using_summary:
            return history.get("history", "")
        else:
            messages = history.get("history", [])
            lines = []
            for msg in messages:
                role = "User" if msg.__class__.__name__ == "HumanMessage" else "Assistant"
                lines.append(f"{role}: {msg.content}")
            return "\n".join(lines)


# Quick test
mem = MemoryManager(llm)
mem.add_turn(
    "We need to file a variation for our cardiovascular product.",
    "Noted. Could you confirm the urgency level?"
)
mem.add_turn(
    "Standard priority — deadline in 3 months.",
    "Thank you. Which team is submitting this?"
)
print("History text:")
print(mem.get_history_text())

## 6. The Core Extraction Logic With Memory

The key design: pass the `collected_fields` dict to the model in every extraction call. The model is told which fields are already confirmed and asked only for what is still missing.

In [ ]:
MULTI_TURN_EXTRACTION_SYSTEM = """
You are SmartIntake, a regulatory affairs intake specialist.
You are in an ongoing conversation collecting five regulatory intake fields.

Fields to collect:
  query_type: complaint | submission | variation | safety_signal | label_update | inspection | general_enquiry
  regulation_ref: FDA_21CFR | EMA_CTR | ICH_E2A | ICH_Q10 | CDSCO_NDC | GxP_GMP | GxP_GCP | other
  product_area: oncology | cardiovascular | infectious_disease | cmc | clinical | labelling | general
  urgency: routine | standard | urgent | critical
  submitting_team: team or function name — never an individual person's name

RULES:
- NEVER ask for a field that is already in 'Already collected fields'.
- Extract any new field values from the latest user message.
- NEVER infer urgency from tone — only from an explicit deadline.
- If the input is not a regulatory query, respond with: {{"error": "non_regulatory_input"}}

Return ONLY a JSON object with the fields you can extract from the latest message.
Do not repeat already-collected fields. Do not invent values.
"""

multi_turn_prompt = ChatPromptTemplate.from_messages([
    ("system", MULTI_TURN_EXTRACTION_SYSTEM),
    ("human",
     "Conversation history:\n{history}\n\n"
     "Already collected fields: {collected_fields}\n\n"
     "Latest message: {user_message}")
])

multi_turn_chain = multi_turn_prompt | llm | StrOutputParser()


# def parse_json_output(text: str) -> dict:
#     """Strip markdown fences and parse JSON."""
#     clean = (
#         text.strip()
#         .removeprefix("```json").removeprefix("```")
#         .removesuffix("```")
#         .strip()
#     )
#     return json.loads(clean)

def parse_json_output(text: str) -> dict:
    """
    Strip markdown code fences and parse the JSON from the LLM's text output.
    
    The model sometimes wraps JSON in ```json ... ``` fences even when instructed
    not to. This function handles both fenced and unfenced output.
    """
    clean = (
        text.strip()
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )
    return json.loads(clean)


print("Multi-turn extraction chain defined.")

## 7. Simulating the Partial Path (T2)

> With Memory

T2 provides `query_type`, `regulation_ref`, and `product_area` but not `urgency` or `submitting_team`. We simulate three turns and verify that already-collected fields are never re-requested.

In [ ]:
# Simulate the T2 partial path across three turns.
# The collected_fields dict accumulates fields turn by turn.
# The model receives the current state on each turn and adds only new fields.

mem = MemoryManager(llm)
# ^^^ MEMORY: Creates a conversation memory manager. Internally this wraps
#     ConversationBufferMemory that stores the full chat history
#     (user messages + assistant responses). On each turn, get_history_text()
#     returns all prior turns as a single string injected into the prompt,
#     so Claude sees the full conversation context, not just the current message.

collected_fields = {}  # Tracks which 5 regulatory fields are already confirmed.
                        # This is NOT memory — it's structured extracted data.
                        # Separate from the chat log stored in mem.

# Simulated user messages for T2 across three turns
simulated_turns = [
    "Hi, we need to file a Type II variation for our cardiovascular product with the EMA.",
    "Standard priority — the filing deadline is in about 3 months.",
    "This is being submitted by the Regulatory Affairs team.",
]

print("=== T2 Partial Path Simulation ===\n")

for turn_number, user_message in enumerate(simulated_turns, start=1):
    print(f"--- Turn {turn_number} ---")
    print(f"User: {user_message}")

    # ===== MEMORY INJECTION POINT =====
    # The multi_turn_chain prompt has three placeholders:
    #
    #   1. {history} — mem.get_history_text() returns the full chat log so far.
    #      On turn 1: empty string (no prior conversation).
    #      On turn 2: "User: ...\nAssistant: Thank you. Could you confirm..."
    #      On turn 3: both prior turns concatenated.
    #      Without this, Claude wouldn't know what it asked or what the user answered.
    #
    #   2. {collected_fields} — the already-confirmed fields as JSON.
    #      This is structured data, not chat history. Tells Claude: "don't re-extract these."
    #
    #   3. {user_message} — the current turn's raw input.
    #
    raw = multi_turn_chain.invoke({
        "history": mem.get_history_text(),           # Full chat history string
        "collected_fields": json.dumps(collected_fields),  # Already extracted data
        "user_message": user_message                 # New message from user
    })

    try:
        new_fields = parse_json_output(raw)
        if new_fields.get("error") == "non_regulatory_input":
            print("FALLBACK: non-regulatory input detected.")
            break
        # Merge new non-null fields into collected_fields
        for k, v in new_fields.items():
            if v is not None:
                collected_fields[k] = v
    except json.JSONDecodeError:
        print(f"  Parse error on turn {turn_number}: {raw[:80]}")

    print(f"Collected so far: {list(collected_fields.keys())}")

    # Determine what is still missing
    all_fields = ["query_type", "regulation_ref", "product_area", "urgency", "submitting_team"]
    missing = [f for f in all_fields if f not in collected_fields]

    if missing:
        follow_up = f"Thank you. To complete the intake record, could you confirm: {', '.join(missing)}?"
        print(f"Assistant asks for: {missing}")

        # ===== MEMORY STORAGE POINT =====
        # mem.add_turn(user_message, follow_up) stores BOTH sides of this
        # conversation turn into the ConversationBufferMemory.
        # This is critical because on the NEXT iteration, get_history_text()
        # will include this turn in its output. Without this call, the history
        # string would remain empty and Claude would forget what it just asked
        # and what the user just answered — making the multi-turn loop useless.
        mem.add_turn(user_message, follow_up)
    else:
        print("All five fields collected.")
        mem.add_turn(user_message, "All fields confirmed.")
        break
    print()

print("\nFinal collected fields:")
print(json.dumps(collected_fields, indent=2))


In [ ]:
# ===== WITHOUT MEMORY: Each turn is a fresh conversation =====
# No MemoryManager. No mem.get_history_text(). No mem.add_turn().
# Each call to multi_turn_chain has empty history, so Claude has
# NO context of what was said in previous turns.

collected_fields = {}  # Still accumulates extracted fields (this is just a dict, not memory)

# Simulated user messages for T2 across three turns
simulated_turns = [
    "Hi, we need to file a Type II variation for our cardiovascular product with the EMA.",
    "Standard priority — the filing deadline is in about 3 months.",
    "This is being submitted by the Regulatory Affairs team.",
]

print("=== T2 WITHOUT MEMORY — Each turn is a fresh start ===\n")

for turn_number, user_message in enumerate(simulated_turns, start=1):
    print(f"--- Turn {turn_number} ---")
    print(f"User: {user_message}")

    # NO history injected — the prompt's {history} placeholder gets an empty string
    # Claude has NO idea what was discussed before. It sees each message in isolation.
    raw = multi_turn_chain.invoke({
        "history": "",                               # ← EMPTY — no memory of past turns!
        "collected_fields": json.dumps(collected_fields),  # Still passes what we've collected
        "user_message": user_message                 # Only sees the current message
    })

    try:
        new_fields = parse_json_output(raw)
        if new_fields.get("error") == "non_regulatory_input":
            print("FALLBACK: non-regulatory input detected.")
            break
        for k, v in new_fields.items():
            if v is not None:
                collected_fields[k] = v
    except json.JSONDecodeError:
        print(f"  Parse error on turn {turn_number}: {raw[:80]}")

    print(f"Collected so far: {list(collected_fields.keys())}")

    all_fields = ["query_type", "regulation_ref", "product_area", "urgency", "submitting_team"]
    missing = [f for f in all_fields if f not in collected_fields]

    if missing:
        follow_up = f"Thank you. To complete the intake record, could you confirm: {', '.join(missing)}?"
        print(f"Assistant asks for: {missing}")
        # NOTE: No mem.add_turn() — we don't store this exchange anywhere
    else:
        print("All five fields collected.")
        break
    print()

print("\nFinal collected fields (WITHOUT memory):")
print(json.dumps(collected_fields, indent=2))


## 8. Log-Safe JSON Output

When all five fields are validated, save the record to a timestamped JSON file. The file must contain only the five structured fields plus metadata — no raw user messages.

In [ ]:
def save_intake_record(record: IntakeRecord, turns_taken: int) -> str:
    """
    Save a validated intake record to a log-safe JSON file.
    
    COMPLIANCE RULE: This function writes only structured fields and metadata.
    The raw user messages from the session are never included.
    The log_safe sentinel confirms this requirement has been met.
    
    Returns the path to the saved file.
    """
    timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    filepath = f"output/intake_{timestamp}.json"

    output_data = {
        **record.model_dump(),           # The five validated fields
        "timestamp": datetime.now().isoformat(),
        "turns_taken": turns_taken,
        "log_safe": True                 # Sentinel: confirms no raw messages were written
    }

    with open(filepath, "w") as f:
        json.dump(output_data, f, indent=2)

    return filepath


# Validate the collected fields and save the record
try:
    record = IntakeRecord(**collected_fields)
    filepath = save_intake_record(record, turns_taken=mem.turn_count)
    print(f"Record saved to: {filepath}")
    print()
    with open(filepath) as f:
        print(f.read())
except ValidationError as e:
    print("Validation failed — record not saved:")
    missing = [str(err["loc"][0]) for err in e.errors()]
    print(f"Missing fields: {missing}")

## 9. Simulating the Fallback Path (T4)

In [ ]:
# The fallback path handles non-regulatory input gracefully.
# No record is saved. The assistant explains what it handles and asks the user to restate.

mem_fallback = MemoryManager(llm)
collected_fallback = {}

non_regulatory_message = "Can you check what the weather is like in Mumbai today?"

print(f"User: {non_regulatory_message}")

raw = multi_turn_chain.invoke({
    "history": mem_fallback.get_history_text(),
    "collected_fields": json.dumps(collected_fallback),
    "user_message": non_regulatory_message
})

try:
    result = parse_json_output(raw)
    if result.get("error") == "non_regulatory_input":
        fallback_response = (
            "SmartIntake handles pharmaceutical regulatory intake queries only. "
            "Please describe your compliance obligation, regulatory deadline, or "
            "agency interaction and I will classify it for you."
        )
        print(f"\nAssistant (fallback): {fallback_response}")
        mem_fallback.add_turn(non_regulatory_message, fallback_response)
    else:
        print(f"Unexpected result: {result}")
except json.JSONDecodeError:
    print(f"Parse error: {raw}")

---
## Summary

You have:
- Set up `ConversationBufferMemory` and `ConversationSummaryMemory`
- Built the `MemoryManager` with automatic switching at 10 turns
- Simulated the partial path (T2) across three turns without re-asking for confirmed fields
- Saved a log-safe JSON record with the `log_safe: true` sentinel
- Demonstrated the graceful fallback path for non-regulatory input (T4)

**Next:** Open `NB-07_integration.ipynb` to assemble all five layers into a complete end-to-end system and prepare for the demo.